# BirdCLEF+ 2025 — EDA + Domain Shift Analysis

Replicates the analysis from Sydorskyi & Gonçalves (2025), Sections 4.1-4.2:
- Species distribution and class imbalance (Figure 1)
- Domain shift analysis: training vs soundscape species distributions
- Multi-species co-occurrence in soundscapes (Table 10)
- Spectrogram comparison: focal vs soundscape recordings (Figure 6)
- Secondary label statistics (Xeno-Canto background species)


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
import soundfile as sf
import torch

from code_base.augmentations.nnaudio_mel import MelExtractor

plt.rcParams.update({'figure.dpi': 150, 'font.size': 11})
sns.set_theme(style='whitegrid')

DATA_ROOT = Path('/data/birdclef-2025')
print('Ready')

## 1. Dataset Overview (Table 1 & 2 from paper)

In [ ]:
df = pd.read_csv(DATA_ROOT / 'train_metadata.csv')
print(f'Total samples: {len(df):,}')
print(f'Unique primary species: {df["primary_label"].nunique()}')
print()

# Table 1: Taxonomic class distribution
if 'class_name' in df.columns:
    print('Taxonomic class distribution (Table 1):')
    print(df.groupby('class_name')[['primary_label']].nunique().rename(columns={'primary_label': 'n_species'}))

# Table 2: Collection source distribution
if 'source' in df.columns:
    print()
    print('Collection source (Table 2):')
    src_counts = df['source'].value_counts()
    src_pct = df['source'].value_counts(normalize=True) * 100
    pd.DataFrame({'Count': src_counts, 'Percentage': src_pct.round(2)}).head(10)

## 2. Class Imbalance (Figure 1 from paper)

In [ ]:
counts = df['primary_label'].value_counts().sort_values()
p25, p75 = counts.quantile([0.25, 0.75])
imbalance_ratio = counts.max() / counts.min()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: sorted bar chart (Figure 1 style)
axes[0].bar(range(len(counts)), counts.values, width=1.0, color='steelblue', alpha=0.8)
axes[0].axhline(p25, color='red',    ls='--', lw=1.5, label=f'25th pct = {p25:.0f}')
axes[0].axhline(p75, color='orange', ls='--', lw=1.5, label=f'75th pct = {p75:.0f}')
axes[0].set_xlabel('Species (sorted by frequency)')
axes[0].set_ylabel('Number of training samples')
axes[0].set_title(f'Training sample distribution (206 species)\nImbalance ratio = {imbalance_ratio:.0f}×')
axes[0].legend()

# Right: histogram
axes[1].hist(counts.values, bins=40, color='steelblue', edgecolor='white')
axes[1].axvline(10, color='red', ls='--', label='<10 = undersampled (39 species)')
axes[1].axvline(500, color='green', ls='--', label='>500 = frequent (9 species)')
axes[1].set_xlabel('Samples per species')
axes[1].set_ylabel('Number of species')
axes[1].set_title('Per-species sample count distribution')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('eda_class_imbalance.png', bbox_inches='tight')
plt.show()

n_under10 = (counts < 10).sum()
n_over500 = (counts >= 500).sum()
print(f'Undersampled (<10 samples): {n_under10} species')
print(f'Frequent (≥500 samples): {n_over500} species')
print(f'Paper reports: 39 species with <10 recordings, 9 with >500')

## 3. Balanced Sampling Weight Comparison (Eq. 1, γ=-0.5 vs γ=-1)

In [ ]:
# Paper Eq. 1: w_i = (c_i / Σc_j)^γ
c = np.sort(counts.values)[::-1]
total = c.sum()
freq = c / total

fig, ax = plt.subplots(figsize=(14, 4))
x = range(len(c))

for gamma, label, color in [
    (0.0,  'Uniform (γ=0)',               'gray'),
    (-0.5, 'SqrtBalancing NFNet (γ=-0.5)', 'steelblue'),
    (-1.0, 'EqualBalancing EffNetV2 (γ=-1)', 'orange'),
]:
    w = freq ** gamma
    w = w / w.sum()
    ax.plot(x, w, label=label, alpha=0.8)

ax.set_xlabel('Species (most frequent → least frequent)')
ax.set_ylabel('Normalized sampling weight w_i')
ax.set_title('Sampling weight strategies — paper Eq. 1: w_i = (c_i / Σc_j)^γ')
ax.legend()
plt.tight_layout()
plt.savefig('eda_sampling_weights.png', bbox_inches='tight')
plt.show()

## 4. Domain Shift: Multi-species co-occurrence (Table 10 from paper)

In [ ]:
# From paper Table 10:
# Training: >90% single-species, max 12 species
# Soundscapes: 54% no species, up to 25 species

# Count secondary labels per training sample
if 'secondary_labels' in df.columns:
    def count_species(row):
        sec = str(row.get('secondary_labels', '')).strip()
        primary = 1  # Always has primary
        if sec and sec not in ('', 'nan', '[]'):
            secondary = len(sec.strip('[]').replace("'", "").split())
            return primary + secondary
        return primary

    train_n_species = df.apply(count_species, axis=1)
else:
    train_n_species = pd.Series([1] * len(df))  # All single-species

# Table 10 data from paper
soundscape_dist = {
    '0': 54.45, '1': 11.00, '2': 7.11, '3': 5.28,
    '4': 4.11,  '5': 3.39,  '6+': 14.66
}

train_dist = train_n_species.value_counts(normalize=True).sort_index() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training distribution
x_train = [str(i) for i in sorted(train_n_species.unique())[:7]]
y_train = [train_dist.get(int(k), 0) for k in x_train]
axes[0].bar(x_train, y_train, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Number of species per recording')
axes[0].set_ylabel('Percentage of recordings (%)')
axes[0].set_title('Training data: species per recording\n(Weak labels — paper §4.2)')

# Soundscape distribution (from paper Table 10)
axes[1].bar(soundscape_dist.keys(), soundscape_dist.values(), color='orange', alpha=0.8)
axes[1].set_xlabel('Number of species per 5-sec chunk')
axes[1].set_ylabel('Percentage of chunks (%)')
axes[1].set_title('Soundscape data: species per 5-sec chunk\n(Strong labels — paper Table 10)')

plt.suptitle('DOMAIN SHIFT: Training (single-species) vs Test (multi-species)', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_domain_shift.png', bbox_inches='tight')
plt.show()

train_single = (train_n_species == 1).mean() * 100
print(f'Training: {train_single:.1f}% single-species (paper: >90%)')
print(f'Soundscapes: 54.45% no species, 11% single species (paper Table 10)')

## 5. Spectrogram Comparison: Focal vs Soundscape (Figure 6 from paper)

In [ ]:
mel = MelExtractor(sr=32000, n_mels=128, fmin=20, n_fft=2048, hop_length=512)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Load sample training clip
sample_train = df.sample(2, random_state=42)
for i, (_, row) in enumerate(sample_train.iterrows()):
    fpath = DATA_ROOT / 'train_audio' / row['filename']
    if fpath.exists():
        audio, sr = sf.read(str(fpath), dtype='float32')
        if audio.ndim > 1: audio = audio.mean(axis=1)
        import librosa
        if sr != 32000: audio = librosa.resample(audio, orig_sr=sr, target_sr=32000)
        chunk = audio[:160000]
        if len(chunk) < 160000: chunk = np.pad(chunk, (0, 160000-len(chunk)))
        
        with torch.no_grad():
            spec = mel(torch.tensor(chunk).unsqueeze(0)).squeeze().numpy()
        
        axes[0][i].imshow(spec, aspect='auto', origin='lower', cmap='viridis')
        axes[0][i].set_title(f'Training: {row["primary_label"]}\n(clean, single-species)', fontsize=10)
        axes[0][i].set_xlabel('Time frames')
        axes[0][i].set_ylabel('Mel bins')

# Load sample soundscape
soundscape_dir = DATA_ROOT / 'unlabeled_soundscapes'
if soundscape_dir.exists():
    soundscapes = list(soundscape_dir.glob('*.ogg'))[:2]
    for i, spath in enumerate(soundscapes):
        audio, sr = sf.read(str(spath), dtype='float32')
        if audio.ndim > 1: audio = audio.mean(axis=1)
        chunk = audio[30*sr: 35*sr]  # 5s from 30s mark
        if len(chunk) < 160000: chunk = np.pad(chunk, (0, 160000-len(chunk)))
        
        with torch.no_grad():
            spec = mel(torch.tensor(chunk).unsqueeze(0)).squeeze().numpy()
        
        axes[1][i].imshow(spec, aspect='auto', origin='lower', cmap='viridis')
        axes[1][i].set_title(f'Soundscape: {spath.name}\n(noisy, multi-species PAM)', fontsize=10)
        axes[1][i].set_xlabel('Time frames')
        axes[1][i].set_ylabel('Mel bins')
else:
    for ax in axes[1]:
        ax.text(0.5, 0.5, 'Soundscapes not downloaded yet', ha='center', va='center', transform=ax.transAxes)

plt.suptitle('Spectrogram Comparison: Training Clips vs Soundscapes (Figure 6)', fontsize=13)
plt.tight_layout()
plt.savefig('eda_spectrogram_comparison.png', bbox_inches='tight')
plt.show()

## 6. Label Smoothing Visualisation (Eq. 2 from paper)

In [ ]:
# Paper Eq. 2: ỹ_i = y_i(1-α) + α/K  where α=0.05, K=206
K = 206
alpha = 0.05

print(f'Label smoothing (α={alpha}, K={K}):')
print(f'  Positive class (y=1): 1×(1-{alpha}) + {alpha}/{K} = {1*(1-alpha) + alpha/K:.4f}')
print(f'  Negative class (y=0): 0×(1-{alpha}) + {alpha}/{K} = {0*(1-alpha) + alpha/K:.5f}')
print()
print(f'Effect: forces model away from extreme confidence, handles noisy weak labels')
print(f'XC weak labels: species may only vocalize for seconds in a multi-minute recording')